<a href="https://colab.research.google.com/github/vxkudire/python-basic-agents/blob/main/SG_Under_the_Hood_Tokens_and_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Under the Hood: How an LLM Actually Works

So far we've called LLMs over an API - a black box. Now let's open the box and run a
**real model on this machine** (no API key needed). We'll watch a sentence get:

1. **Tokenized** - broken into subword tokens
2. **Converted to numbers** - each token becomes an integer id
3. **Passed through the model** - which outputs numbers (probabilities) for the next token
4. **Decoded back** - numbers -> tokens -> text

We use **SmolLM2-135M-Instruct** (135M parameters, from Hugging Face). It's tiny, free,
and downloads in seconds - the same architecture as GPT-5 or Claude, just far smaller.

In [ ]:
!pip install -q transformers torch

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "HuggingFaceTB/SmolLM2-135M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()
print(f"Loaded {model_name}: {model.num_parameters():,} parameters")

config.json:   0%|          | 0.00/861 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  269MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Loaded HuggingFaceTB/SmolLM2-135M-Instruct: 134,515,008 parameters


## Step 1: Text -> Tokens

The model can't read letters. The **tokenizer** splits text into subword tokens.
Notice the leading spaces (` capital`) - that's how word boundaries are marked.

In [ ]:
text = "The capital of France is"

ids = tokenizer.encode(text)
tokens = [tokenizer.decode([i]) for i in ids]

print("Text  :", text)
print("Tokens:", tokens)
print("Number of tokens:", len(ids))

Text  : The capital of France is
Tokens: ['The', ' capital', ' of', ' France', ' is']
Number of tokens: 5


## Step 2: Tokens -> Numbers

Each token is an entry in the model's vocabulary, identified by an integer **id**.
This is all the model ever sees: a list of integers.

In [ ]:
for token, i in zip(tokens, ids):
    print(f"  {token!r:12} -> {i}")

  'The'        -> 504
  ' capital'   -> 3575
  ' of'        -> 282
  ' France'    -> 4649
  ' is'        -> 314


Let's actually SEE the split. This helper shows each word broken into its token
**pieces** (with `|` between them) and their ids. Common words are often one token;
rarer strings get chopped into subword pieces.

In [ ]:
def show_tokens(text):
    ids = tokenizer.encode(text)
    pieces = [tokenizer.decode([i]) for i in ids]
    print(f"{text!r}")
    print("   split :", " | ".join(pieces))
    print("   ids   :", ids)
    print(f"   -> {len(ids)} token(s)\n")

for w in ["hello", "tokenization", "antidisestablishmentarianism", "GPT"]:
    show_tokens(w)

'hello'
   split : hello
   ids   : [28120]
   -> 1 token(s)

'tokenization'
   split : token | ization
   ids   : [8087, 1237]
   -> 2 token(s)

'antidisestablishmentarianism'
   split : ant | idis | establish | ment | arianism
   ids   : [403, 17889, 30834, 358, 35050]
   -> 5 token(s)

'GPT'
   split : GPT
   ids   : [48474]
   -> 1 token(s)



### Live: how does the model split YOUR name?

Type a few first names from the room into the list below and run the cell. Watch how
differently they break up - a common name may be a single token, while another splits
into 3-4 pieces. (That difference reflects what was common in the training data.)

In [ ]:
names = ["Madhavi", "Venkatesh", "Mani"]   # <- replace with names from the class

for name in names:
    show_tokens(name)

'Madhavi'
   split : Mad | hav | i
   ids   : [24405, 6921, 89]
   -> 3 token(s)

'Venkatesh'
   split : Ven | k | ates | h
   ids   : [38103, 91, 660, 88]
   -> 4 token(s)

'Mani'
   split : Man | i
   ids   : [7764, 89]
   -> 2 token(s)



## Step 3: Numbers -> Model -> Numbers

We feed the ids through the model. It outputs a score (**logit**) for every token in its
vocabulary - its guess for what comes *next*. We softmax those into probabilities and look
at the top 5. Predicting the next token is the model's entire job.

In [ ]:
encoded = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    logits = model(**encoded).logits

next_token_logits = logits[0, -1]
probs = torch.softmax(next_token_logits, dim=-1)
top5 = torch.topk(probs, 5)

print(f"'{text}' ...\n")
print("The model's top-5 guesses for the NEXT token:")
for prob, idx in zip(top5.values, top5.indices):
    print(f"  {tokenizer.decode([idx])!r:12} -> {float(prob):.1%}")

'The capital of France is' ...

The model's top-5 guesses for the NEXT token:
  ' Paris'     -> 44.5%
  ' the'       -> 27.0%
  ' located'   -> 8.8%
  ' called'    -> 2.0%
  ' Le'        -> 1.6%


The model puts most of its probability on ` Paris` - it learned that from training data.

## Step 4: Loop it (generate), then decode back to text

'Generation' is just Step 3 over and over: predict the next token, append it, predict
again. Then decode the final list of ids back into text.

In [ ]:
output_ids = model.generate(
    **encoded,
    max_new_tokens=10,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id,
)

print("Final ids :", output_ids[0].tolist())
print("Final text:", tokenizer.decode(output_ids[0]))

Final ids : [504, 3575, 282, 4649, 314, 7042, 30, 7042, 314, 253, 1739, 2240, 281, 4649, 28]
Final text: The capital of France is Paris. Paris is a major city in France,


## Bonus: this is an INSTRUCT model - the chat template

SmolLM2 is *instruction-tuned* to act like an assistant. To use it that way, we wrap our
question in a **chat template** - special tokens (`<|im_start|>`, roles) the model was
trained on. This is exactly what the OpenAI/Anthropic APIs do for you behind the scenes.

In [ ]:
messages = [{"role": "user", "content": "What is the capital of France? Answer in one sentence."}]

prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
print("What the model actually receives:\n")
print(prompt)

What the model actually receives:

<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
What is the capital of France? Answer in one sentence.<|im_end|>
<|im_start|>assistant



In [ ]:
enc = tokenizer(prompt, return_tensors="pt")
out = model.generate(**enc, max_new_tokens=40, do_sample=False, pad_token_id=tokenizer.eos_token_id)
answer = tokenizer.decode(out[0][enc.input_ids.shape[1]:], skip_special_tokens=True)
print("Assistant:", answer.strip())

Assistant: The capital of France is Paris.


## That's the whole trick

An LLM is a next-token predictor. Text becomes numbers, the model turns numbers into
probabilities for the next token, and we loop and decode. Instruction-tuning + a chat
template make it behave like an assistant. GPT-5 and Claude do exactly this - just with
far more parameters, more training, and much longer context.

**Try changing `text` (Step 1) and the `messages` question (Bonus) and re-run** - watch
the tokens and predictions change.

---
## Fun demos: predict & complete

Two reusable helpers for playing with the model live.

### Demo 1: `next_tokens(phrase)` - what does the model think comes next?

Shows the model's top-5 guesses for the very next token, with a probability bar.
Great for audience phrases like `"Shubham is a"` or `"The best programming language is"`.

In [ ]:
def next_tokens(phrase, k=5):
    enc = tokenizer(phrase, return_tensors="pt")
    with torch.no_grad():
        logits = model(**enc).logits
    probs = torch.softmax(logits[0, -1], dim=-1)
    top = torch.topk(probs, k)
    print(f"{phrase!r} ...\n")
    for prob, idx in zip(top.values, top.indices):
        piece = tokenizer.decode([idx])
        bar = "#" * int(float(prob) * 40)
        print(f"   {piece!r:14} {float(prob):5.1%}  {bar}")

In [ ]:
next_tokens("Shubham is a")            # <- try an audience name
next_tokens("My favorite food is")
next_tokens("The best programming language is")

'Shubham is a' ...

   ' popular'      3.0%  #
   ' small'        2.4%  
   ' place'        2.4%  
   ' city'         2.1%  
   ' village'      1.8%  
'My favorite food is' ...

   ' definitely'   9.7%  ###
   ' sushi'        5.2%  ##
   ' always'       5.2%  ##
   ' a'            5.2%  ##
   ' pizza'        4.6%  #
'The best programming language is' ...

   ' Python'      10.7%  ####
   ' the'         10.7%  ####
   ' not'          7.4%  ##
   ' one'          5.1%  ##
   ' what'         3.1%  #


> **Tip:** don't put a trailing space (`"Shubham is a "`). A space changes the last token
> and the model starts predicting digits - a neat reminder that the model sees *tokens*,
> not words. Try it both ways and show the class the difference!

### Demo 2: `complete(phrase)` - let the model finish the sentence

Generates the next several words. The **temperature** controls randomness:
- `temperature=0` -> greedy & **deterministic** (same output every time)
- higher (0.7-1.0) -> more varied and creative
- very high (1.5+) -> wild, sometimes nonsense

In [ ]:
def complete(phrase, max_new_tokens=20, temperature=0.0):
    enc = tokenizer(phrase, return_tensors="pt")
    if temperature <= 0:
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    else:
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=True,
                             temperature=temperature, top_p=0.95,
                             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0], skip_special_tokens=True)

In [ ]:
# Deterministic: run this a few times - the output never changes.
print(complete("Shubham is a", max_new_tokens=20))

Shubham is a popular destination for tourists and locals alike. The city is known for its vibrant nightlife, delicious cuisine


Now crank the temperature and run the SAME phrase 3 times - watch it change every time:

In [ ]:
for i in range(3):
    print(f"[{i+1}] {complete('Shubham is a', max_new_tokens=20, temperature=0.8)}")

[1] Shubham is a big, interesting town with a rich history and a well-preserved medieval architecture.

To get
[2] Shubham is a famous restaurant in Bangalore. The restaurant's owner, Rani Lakshmi Bhattacharya,
[3] Shubham is a popular destination in the region.


And a very high temperature - the model gets wild (and often nonsensical):

In [ ]:
for i in range(2):
    print(f"[{i+1}] {complete('Shubham is a', max_new_tokens=20, temperature=1.5)}")

[1] Shubham is a local village just north of Patti, and they all went hunting, some going to hunt deer by
[2] Shubham is a famous actor and dancer named Shah Nayi Shah who also teaches yoga poses there. Shah Nay


**That's temperature in one demo:** 0 = the model's single most likely path (repeatable);
higher = it samples from its other guesses, so you get variety. In real apps you use low
temperature for classification/extraction and higher for brainstorming or creative text.

---
# Phase 2: Use the LLM as a Chatbot

We poked at a tiny model to *see the guts*. Now we switch to a bigger model over the
**OpenAI API** to actually build something. Same idea (next-token prediction) - just a
much larger model, and OpenAI handles the tokenizing + chat template for us.

Set `OPENAI_API_KEY` as a Colab secret.

In [ ]:
from openai import OpenAI
from google.colab import userdata

client = OpenAI(api_key=userdata.get('OPENAI_API_KEY_NEW'))
MODEL = "gpt-5.4-mini"

### The key idea: memory is just a growing list

A chatbot 'remembers' because we keep every turn in a `messages` list and send the
**whole list** each time. Watch: tell it your name, then ask what your name is.

In [ ]:
chat_history = [{"role": "system", "content": "You are a friendly, concise assistant."}]

def chat(user_message):
    chat_history.append({"role": "user", "content": user_message})
    response = client.chat.completions.create(model=MODEL, messages=chat_history)
    reply = response.choices[0].message.content
    chat_history.append({"role": "assistant", "content": reply})
    return reply

A simple chat box (runs in Colab). Type a message and click **Send**:

In [ ]:
import ipywidgets as widgets
from IPython.display import display

output = widgets.Output()
text_input = widgets.Text(placeholder="Type your message...", layout=widgets.Layout(width="70%"))
send_button = widgets.Button(description="Send", button_style="primary")

def on_send(_):
    msg = text_input.value.strip()
    if not msg:
        return
    text_input.value = ""
    with output:
        print(f"You:       {msg}")
        print(f"Assistant: {chat(msg)}\n")

send_button.on_click(on_send)
display(widgets.VBox([output, widgets.HBox([text_input, send_button])]))

**If the chat box above doesn't appear**, run this plain-text version instead
(type `quit` to stop):

In [ ]:
while True:
    msg = input("You: ")
    if msg.lower() in ("quit", "exit"):
        break
    print("Assistant:", chat(msg), "\n")

KeyboardInterrupt: Interrupted by user

**That's the whole trick to memory:** every turn is appended to `chat_history`, and we
resend the full list. The model has no memory of its own - the *list* is the memory.
(This is the 'short-term memory / context window' idea from the slides.)

---
# Phase 3: Turn the LLM into an Agent

A chatbot only *talks*. An **agent** can *act* - by calling tools. We'll give the model
two tools and watch it decide when to use them.

### ⚠️ The #1 misconception to clear up
People think **the LLM calls the tool and gets the answer**. It does NOT. The LLM can
only *ask* for a tool. **Our Python code** actually runs it and hands the result back.
The steps below make this explicit.

### The two tools (plain Python functions, tested & reliable)

In [ ]:
import requests, urllib.parse

def get_weather(city):
    # 1) city name -> latitude/longitude (Open-Meteo geocoding, keyless)
    geo = requests.get("https://geocoding-api.open-meteo.com/v1/search",
                       params={"name": city, "count": 1}, timeout=10).json()
    if not geo.get("results"):
        return f"Could not find a city called {city!r}."
    loc = geo["results"][0]
    # 2) lat/lon -> current weather
    w = requests.get("https://api.open-meteo.com/v1/forecast",
                     params={"latitude": loc["latitude"], "longitude": loc["longitude"],
                             "current": "temperature_2m,wind_speed_10m"}, timeout=10).json()
    cur = w["current"]
    return (f"{loc['name']}, {loc.get('country','')}: "
            f"{cur['temperature_2m']}C, wind {cur['wind_speed_10m']} km/h")

def search_wikipedia(query):
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{urllib.parse.quote(query)}"
    r = requests.get(url, headers={"User-Agent": "ik-demo/1.0"}, timeout=10)
    if r.status_code != 200:
        return f"No Wikipedia article found for {query!r}."
    return r.json().get("extract", "No summary available.")

AVAILABLE_TOOLS = {"get_weather": get_weather, "search_wikipedia": search_wikipedia}

# quick sanity check (this is US, not the LLM, calling them):
print(get_weather("Tokyo"))
print(search_wikipedia("Eiffel Tower")[:100])

Tokyo, Japan: 22.3C, wind 4.7 km/h
The Eiffel Tower is a lattice tower on the Champ de Mars in Paris, France. It is named after the eng


### Describe the tools to the model (the schema = the tool's menu)

In [ ]:
tools_schema = [
    {"type": "function", "function": {
        "name": "get_weather",
        "description": "Get the current weather for a city.",
        "parameters": {"type": "object",
            "properties": {"city": {"type": "string", "description": "City name, e.g. 'Tokyo'"}},
            "required": ["city"]}}},
    {"type": "function", "function": {
        "name": "search_wikipedia",
        "description": "Look up a short summary of a topic from Wikipedia.",
        "parameters": {"type": "object",
            "properties": {"query": {"type": "string", "description": "The topic to look up."}},
            "required": ["query"]}}},
]

### The agent loop - with every step made visible

Read the printed steps when you run it. You'll SEE that the LLM only *requests* a tool;
our Python code runs it and feeds the result back.

In [ ]:
import json

def run_agent(user_question, max_turns=5):
    messages = [{"role": "user", "content": user_question}]
    print(f"USER: {user_question}\n")

    for _ in range(max_turns):
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=tools_schema, tool_choice="auto")
        msg = response.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:
            print(f"LLM (final answer): {msg.content}")
            return

        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments)
            print(f"STEP 1  LLM asks to call: {name}({args})")
            print(f"        (the LLM did NOT run it - it only asked)")

            result = AVAILABLE_TOOLS[name](**args)   # <- OUR Python code runs the tool
            print(f"STEP 2  PYTHON ran {name} -> {result}")
            print(f"STEP 3  we hand that result back to the LLM\n")

            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "name": name, "content": str(result)})
    print("Stopped after max_turns.")

### Try it - watch the steps

In [ ]:
run_agent("can you tell me weather in Boston in Farhenheit and miles/hour")

USER: can you tell me weather in Boston in Farhenheit and miles/hour

STEP 1  LLM asks to call: get_weather({'city': 'Boston'})
        (the LLM did NOT run it - it only asked)
STEP 2  PYTHON ran get_weather -> Boston, United States: 30.5C, wind 20.6 km/h
STEP 3  we hand that result back to the LLM

LLM (final answer): Boston weather: **30.5°C** and wind **20.6 km/h**.

Converted:
- **Temperature:** **86.9°F**
- **Wind:** **12.8 mph**

If you want, I can also format it as a quick “feels like” style weather report.


In [ ]:
run_agent("Give me a one-line summary of the Eiffel Tower.")

USER: Give me a one-line summary of the Eiffel Tower.

LLM (final answer): The Eiffel Tower is an iron lattice tower in Paris, built in 1889, and one of the world’s most recognizable landmarks.


In [ ]:
run_agent("Can you search wikipedia on Eiffer tower and give me one line summary afterwards")

USER: Can you search wikipedia on Eiffer tower and give me one line summary afterwards

STEP 1  LLM asks to call: search_wikipedia({'query': 'Eiffel Tower'})
        (the LLM did NOT run it - it only asked)
STEP 2  PYTHON ran search_wikipedia -> The Eiffel Tower is a lattice tower on the Champ de Mars in Paris, France. It is named after the engineer Gustave Eiffel, whose company designed and built the tower from 1887 to 1889.
STEP 3  we hand that result back to the LLM

LLM (final answer): The Eiffel Tower is a lattice tower in Paris, France, designed and built by Gustave Eiffel’s company from 1887 to 1889.


In [ ]:
run_agent("What's the weather in Paris, and what is Paris famous for?")   # may use BOTH tools

**The takeaway:** the LLM is the *decider* - it reads the question and chooses which tool
to call. But it never touches the weather API or Wikipedia. **Our code (the orchestrator)**
executes the tool and returns the data; then the LLM writes the final answer from that data.
That LLM-decides / code-executes loop is exactly what makes an agent.